# 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

In [ ]:
!pip install koreanize_matplotlib -q
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import koreanize_matplotlib

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # 경고 메시지 감추기
import matplotlib.font_manager as fm
# 1. 나눔폰트 자동 설치 및 폰트 매니저 재설정
!apt-get install -y fonts-nanum > /dev/null
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')

# 2. 전역 폰트 지정
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지


# 1. 고위험 유저 그룹(Target Group)

## 데이터 타입 확인

In [ ]:
%%bigquery --project your-gcp-project
SELECT table_name, column_name, data_type
FROM `your-gcp-project.stage.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name IN ('stg_vote', 'stg_users', 'stg_user')
ORDER BY table_name, ordinal_position;

## 고위험 유저 그룹(Target Group) 정의

In [ ]:
%%bigquery df_risk_users --project your-gcp-project
WITH user_risk_metrics AS (
  SELECT
    user_id,
    ban_status,
    COALESCE(report_count, 0) AS report_cnt,
    COALESCE(block_count, 0) AS block_cnt,
    COALESCE(hide_count, 0) AS hide_cnt,
    -- 신고 + 차단 합산 (실제 타인에게 피해를 주거나 제재받은 수치)
    (COALESCE(report_count, 0) + COALESCE(block_count, 0)) AS direct_risk_score,
    -- 전체 활동 문제 수치 (숨김 포함)
    (COALESCE(report_count, 0) + COALESCE(block_count, 0) + COALESCE(hide_count, 0)) AS total_risk_score
  FROM `your-gcp-project.stage.stg_users`
  WHERE is_valid_user = TRUE
)
SELECT
  user_id,
  ban_status,
  report_cnt,
  block_cnt,
  hide_cnt,
  direct_risk_score,
  total_risk_score,
  -- 정교하게 정제된 위험군 분류
  CASE
    WHEN ban_status IS NOT NULL AND ban_status NOT IN ('', 'N') THEN 'High Risk (실제 제재/탈퇴)'
    WHEN (report_cnt + block_cnt) >= 3 THEN 'High Risk (고신고/고차단)'
    WHEN (report_cnt + block_cnt) >= 1 THEN 'Medium Risk (주의)'
    WHEN hide_cnt >= 10 THEN 'Heavy Hide User (숨김 다수 사용)'
    ELSE 'Normal (일반)'
  END AS risk_group
FROM user_risk_metrics
ORDER BY direct_risk_score DESC, hide_cnt DESC;

## 각 위험 그룹별 유저 수 분포

1. ban_status 조건 정제: 'N'이 아닌 실제 제재 상태만 제재군으로 분류

2. risk_group 세분화

- Normal (일반): 634,627명
- Medium Risk (주의) : 신고/차단 수치가 다소 있거나 모니터링이 필요한 유저 / 25,294명
- Heavy Hide User (숨김 다수 사용): 숨김 건수만 높고 신고는 적은 유저 / 5,822명
- High Risk (고신고/고차단), 피해/제재 위험군: 신고당함 + 차단당함 수치가 매우 높거나 실제 제재 대상인 유저 / 2,689명

In [ ]:
# 정제된 위험군 그룹별 유저 수 확인
print("=== 정제된 위험군 그룹별 분포 ===")
print(df_risk_users['risk_group'].value_counts())

# 고신고/고차단 유저 상위 확인
df_risk_users[df_risk_users['risk_group'].str.contains('High Risk|Medium Risk')].head(10)

# 2. 유저 그룹별 특징 비교 분석

In [ ]:
%%bigquery df_risk_validation --project your-gcp-project
WITH base_risk_users AS (
  SELECT
    user_id,
    CASE
      WHEN ban_status IS NOT NULL AND ban_status NOT IN ('', 'N') THEN 'High Risk (제재/탈퇴)'
      WHEN (COALESCE(report_count, 0) + COALESCE(block_count, 0)) >= 3 THEN 'High Risk (고신고/고차단)'
      WHEN (COALESCE(report_count, 0) + COALESCE(block_count, 0)) >= 1 THEN 'Medium Risk (주의)'
      WHEN COALESCE(hide_count, 0) >= 10 THEN 'Heavy Hide User (숨김 다수)'
      ELSE 'Normal (일반)'
    END AS risk_group
  FROM `your-gcp-project.stage.stg_users`
  WHERE is_valid_user = TRUE
),
payment_summary AS (
  SELECT user_id, COUNT(*) AS pay_cnt FROM `your-gcp-project.stage.stg_payment` GROUP BY user_id
)
SELECT
  r.risk_group,
  COUNT(DISTINCT r.user_id) AS total_users,
  -- 실제 결제 경험이 있는 유저 비율 (%)
  ROUND(COUNT(DISTINCT CASE WHEN pm.pay_cnt > 0 THEN r.user_id END) * 100.0 / COUNT(DISTINCT r.user_id), 2) AS paying_user_ratio_pct,
  -- 중앙값 (Median) 확인
  APPROX_QUANTILES(COALESCE(pm.pay_cnt, 0), 100)[OFFSET(50)] AS median_payment_cnt
FROM base_risk_users r
LEFT JOIN payment_summary pm ON r.user_id = pm.user_id
GROUP BY r.risk_group
ORDER BY total_users DESC;

In [ ]:
# 검증 결과 데이터프레임 출력
print("=== 유저 그룹별 실제 결제 유저 비율 및 중앙값 ===")
df_risk_validation

1. 중앙값(median_payment_cnt)은 전 그룹 모두 0

- 일반 유저든 고위험 유저든 유저 50% 이상(과반수)은 결제를 단 1건도 하지 않았습니다.


2. 실제 결제자 비율(paying_user_ratio_pct)의 한계

- High Risk (고신고/고차단): 결제 유저 비율이 17.11% (약 460명)

- Normal (일반): 결제 유저 비율은 8.44%이지만, 유저 수가 압도적이어서 약 53,500명

결론: 고위험군 유저의 결제 경험 비율(17.11%)이 상대적으로 높아 보이지만,<mark> 실제 서비스 전체 결제 매출을 견인하는 건 **99% 이상의 절대다수를 차지하는 일반 유저**입니다.

## 피해/제재 위험군 vs 숨김 헤비유저 비교

In [ ]:
%%bigquery df_risk_group_comparison --project your-gcp-project
WITH base_users AS (
  SELECT
    user_id,
    ban_status,
    COALESCE(report_count, 0) AS report_cnt,
    COALESCE(block_count, 0) AS block_cnt,
    COALESCE(hide_count, 0) AS hide_cnt,
    COALESCE(point, 0) AS user_point,
    -- 위험군 그룹 세분화 정의
    CASE
      WHEN (COALESCE(report_count, 0) + COALESCE(block_count, 0)) >= 3 THEN '🚨 High Risk (신고/차단 당함)'
      WHEN COALESCE(hide_count, 0) >= 10 THEN '👁️ Heavy Hide User (내가 많이 숨김)'
      ELSE '🟢 Normal (일반)'
    END AS risk_group
  FROM `your-gcp-project.stage.stg_users`
  WHERE is_valid_user = TRUE
)
SELECT
  risk_group,
  COUNT(DISTINCT user_id) AS total_users,
  -- 평균 포인트
  ROUND(AVG(user_point), 1) AS avg_point,
  -- 평균 신고 당한 횟수
  ROUND(AVG(report_cnt), 1) AS avg_reported_cnt,
  -- 평균 차단 당한 횟수
  ROUND(AVG(block_cnt), 1) AS avg_blocked_cnt,
  -- 평균 숨김 처리 횟수
  ROUND(AVG(hide_cnt), 1) AS avg_hide_cnt,
  -- 실제 계정 제재/탈퇴 비율 (%)
  ROUND(COUNTIF(ban_status IS NOT NULL AND ban_status NOT IN ('', 'N')) * 100.0 / COUNT(*), 2) AS banned_ratio_pct
FROM base_users
GROUP BY risk_group
ORDER BY total_users DESC;

In [ ]:
# 비교 결과 데이터프레임 출력
print("=== 피해/제재 위험군 vs 숨김 헤비유저 상세 비교 ===")
df_risk_group_comparison

| 구분 | 🟢 Normal (일반) | 👁️ Heavy Hide User (숨김 다수) | 🚨 High Risk (신고/차단 당함) |
| :--- | :---: | :---: | :---: |
| **유저 수** | 659,184명 (98.6%) | 6,559명 (1.0%) | 2,689명 (0.4%) |
| **평균 포인트** | 3,072.1 p | 2,352.9 p | 2,420.1 p |
| **평균 신고 당한 수 (avg_reported_cnt)** | 0.0회 | 0.0회 | 3.3회 🔥 |
| **평균 차단 당한 수 (avg_blocked_cnt)** | 0.0회 | 0.1회 | 2.2회 🔥 |
| **평균 숨김 처리 수 (avg_hide_cnt)** | 0.2회 | 54.1회 👁️ | 4.2회 |

💡 두 집단의 3대 명확한 특징 및 차점
1. **🚨 High Risk (신고/차단 당함): 타인에게 피해를 주는 마찰 유발형**
- 핵심 수치: 평균 신고당함 3.3회, 차단당함 2.2회로 타 집단(0.0회) 대비 압도적인 수치를 보입니다.

- 행동 성격: 부적절한 언행, 자극적 질문 작성, 도배 등으로 타 유저들에게 직접적인 불쾌감과 마찰을 유발하는 집단입니다.

- 비중: 전체 유저의 0.4% (2,689명)로 매우 극소수이지만, 서비스 생태계 전체의 품질을 떨어뜨리는 핵심 어뷰징/리스크 그룹입니다.

2. **👁️ Heavy Hide User (내가 많이 숨김): 피드를 정리하는 충성/개인주의형**
- 핵심 수치: 평균 숨김 처리 횟수 54.1회로 엄청나게 높지만, 타인에게 신고당하거나 차단당한 횟수는 0.0회로 일반 유저와 똑같습니다.

- 행동 성격: 남에게 피해를 주거나 마찰을 일으키는 유저가 아닙니다! 앱을 활발히 사용하면서 관심 없는 질문이나 보기 싫은 피드를 스스로 정리(필터링)하며 사용하는 적극적인 피드 관리형 열성 유저입니다.


🚀 전략적 시사점 (Action Item)
"운영 지표 설정 시 **'숨김(Hide)' 수치는 '신고/차단(Report/Block)' 수치와 반드시 분리**해야 합니다."

**High Risk (신고/차단):**

- 신고 3회 또는 차단 2회 이상 누적 시 경고 모니터링 및 FDS 자동 제재 대상으로 분류

**Heavy Hide User (숨김):**

- 제재 대상이 아닌 열성 유저이므로, 피드 숨김 기능을 더 유연하게 제공하거나 관심사 기반 추천 알고리즘을 개편하여 피드 피로도를 줄여주는 UX 개선을 적용

## 1. 질문 유형(stg_question) 및 마찰 콘텐츠 분석
고위험 유저들이 어떤 질문에 투표하거나 작성할 때 신고/차단을 가장 많이 유발했는지 파악

### 유저 그룹별 질문 응답/투표 분포 보기 (전체)

In [ ]:
import pandas as pd
from google.cloud import bigquery

# Pandas에서 질문 텍스트가 잘리지 않도록 설정
pd.set_option('display.max_colwidth', None)

client = bigquery.Client(project='your-gcp-project')

query_all = """
WITH user_groups AS (
    SELECT
        user_id,
        CASE
            WHEN report_count >= 3 OR block_count >= 3 THEN 'High Risk'
            WHEN hide_count >= 5 THEN 'Heavy Hide User'
            ELSE 'Normal'
        END AS user_group
    FROM `your-gcp-project.stage.stg_users`
)
SELECT
    q.question_id,
    q.question_text,
    ug.user_group,
    COUNT(v.vote_id) AS vote_count
FROM `your-gcp-project.stage.stg_question` q
INNER JOIN `your-gcp-project.stage.stg_vote` v ON q.question_id = v.question_id
INNER JOIN user_groups ug ON v.sender_user_id = ug.user_id
GROUP BY 1, 2, 3
ORDER BY vote_count DESC
LIMIT 15;
"""

df_all = client.query(query_all).to_dataframe()
df_all

### 고위험군(High Risk) 유저만 답한 Top 10 질문 보기

In [ ]:
query_high_risk = """
WITH user_groups AS (
    SELECT
        user_id,
        CASE
            WHEN report_count >= 3 OR block_count >= 3 THEN 'High Risk'
            WHEN hide_count >= 5 THEN 'Heavy Hide User'
            ELSE 'Normal'
        END AS user_group
    FROM `your-gcp-project.stage.stg_users`
)
SELECT
    q.question_id,
    q.question_text,
    COUNT(v.vote_id) AS high_risk_vote_count
FROM `your-gcp-project.stage.stg_question` q
INNER JOIN `your-gcp-project.stage.stg_vote` v ON q.question_id = v.question_id
INNER JOIN user_groups ug ON v.sender_user_id = ug.user_id
WHERE ug.user_group = 'High Risk'
GROUP BY 1, 2
ORDER BY high_risk_vote_count DESC
LIMIT 10;
"""

df_high_risk = client.query(query_high_risk).to_dataframe()
df_high_risk

#### 질문별 유저 그룹 비중(%) 분석 & 시각화 코드

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery

client = bigquery.Client(project='your-gcp-project')

# 1. 질문별 / 위험 그룹별 투표 수 집계 쿼리
query_ratio = """
WITH user_groups AS (
    SELECT
        user_id,
        CASE
            WHEN report_count >= 3 OR block_count >= 3 THEN 'High Risk'
            WHEN hide_count >= 5 THEN 'Heavy Hide User'
            ELSE 'Normal'
        END AS user_group
    FROM `your-gcp-project.stage.stg_users`
)
SELECT
    q.question_id,
    q.question_text,
    ug.user_group,
    COUNT(v.vote_id) AS vote_count
FROM `your-gcp-project.stage.stg_question` q
INNER JOIN `your-gcp-project.stage.stg_vote` v ON q.question_id = v.question_id
INNER JOIN user_groups ug ON v.sender_user_id = ug.user_id
GROUP BY 1, 2, 3
"""

df_ratio_raw = client.query(query_ratio).to_dataframe()

# 2. 피봇 테이블 생성 (행: 질문, 열: 유저그룹, 값: 투표수)
df_pivot = df_ratio_raw.pivot_table(
    index=['question_id', 'question_text'],
    columns='user_group',
    values='vote_count',
    aggfunc='sum',
    fill_value=0
).reset_index()

# 3. 전체 투표 수 및 고위험군 비중(%) 계산
df_pivot['Total_Votes'] = df_pivot.get('High Risk', 0) + df_pivot.get('Heavy Hide User', 0) + df_pivot.get('Normal', 0)

# 신뢰성을 위해 최소 100건 이상 투표가 발생한 질문만 필터링
df_filtered = df_pivot[df_pivot['Total_Votes'] >= 100].copy()
df_filtered['High_Risk_Ratio(%)'] = (df_filtered['High Risk'] / df_filtered['Total_Votes']) * 100

# High Risk 비중이 높은 상위 10개 질문 추출
df_top_risk_ratio = df_filtered.sort_values(by='High_Risk_Ratio(%)', ascending=False).head(10)

# 4. 시각화 (가로 막대 그래프)
plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_top_risk_ratio,
    x='High_Risk_Ratio(%)',
    y='question_text',
    palette='Reds_r'
)

plt.title('High Risk(고신고/고차단) 유저의 응답 비중이 높은 질문 Top 10', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('High Risk 유저 응답 비중 (%)', fontsize=12)
plt.ylabel('질문 내용', fontsize=12)

# 막대 옆에 정확한 % 값 표시
for index, row in enumerate(df_top_risk_ratio.iterrows()):
    val = row[1]['High_Risk_Ratio(%)']
    plt.text(val + 0.1, index, f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# 데이터 프레임으로 상세 숫자 출력
df_top_risk_ratio[['question_id', 'question_text', 'High Risk', 'Total_Votes', 'High_Risk_Ratio(%)']]

## 2. 탈퇴 여부 및 탈퇴 사유(stg_withdraw) 분석
두 그룹 유저들의 실제 탈퇴율과 탈퇴 시 작성한 주요 사유를 비교

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery

# Pandas 텍스트 제한 해제
pd.set_option('display.max_colwidth', None)

client = bigquery.Client(project='your-gcp-project')

# 1. 전체 탈퇴 사유(stg_withdraw) 집계
query_withdraw_reasons = """
SELECT
    COALESCE(reason, '사유 미작성') AS withdraw_reason,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS ratio_pct
FROM `your-gcp-project.stage.stg_withdraw`
GROUP BY 1
ORDER BY count DESC;
"""

df_reasons = client.query(query_withdraw_reasons).to_dataframe()

print("=== 📝 전체 탈퇴 사유 분포 (stg_withdraw) ===")
display(df_reasons)

# 2. stg_users 기준 위험 그룹별 제재/비활성 유저 비율 분석
query_users_status = """
WITH user_groups AS (
    SELECT
        user_id,
        CASE
            WHEN (COALESCE(report_count, 0) + COALESCE(block_count, 0)) >= 3 THEN 'High Risk (고신고/고차단)'
            WHEN COALESCE(hide_count, 0) >= 10 THEN 'Heavy Hide User (숨김 다수)'
            ELSE 'Normal (일반)'
        END AS risk_group,
        ban_status
    FROM `your-gcp-project.stage.stg_users`
)
SELECT
    risk_group,
    COUNT(user_id) AS total_users,
    COUNT(CASE WHEN ban_status IS NOT NULL AND ban_status NOT IN ('', 'N') THEN user_id END) AS banned_users,
    ROUND(COUNT(CASE WHEN ban_status IS NOT NULL AND ban_status NOT IN ('', 'N') THEN user_id END) * 100.0 / COUNT(user_id), 2) AS banned_ratio_pct
FROM user_groups
GROUP BY 1
ORDER BY total_users DESC;
"""

df_user_status = client.query(query_users_status).to_dataframe()

print("\n=== 📊 위험 그룹별 제재/정지 유저 비율 (stg_users) ===")
display(df_user_status)

📊 탈퇴 사유 및 유저 상태 분석 결과 요약
1️⃣ 전체 탈퇴 사유 분석 (stg_withdraw)
- 기타 이유 (56.95%, 40,301건): 절반 이상이 구체적 사유 없이 '기타 이유'를 선택하여 탈퇴함.

- 서비스/콘텐츠 불만 (38.98%):

함께 할 친구가 없어서 (20.42%, 14,450건) ➔ SNS 특성상 초기 유저 유입 및 네트워크 형성(온보딩) 부재가 탈퇴의 가장 큰 원인.

재밌는 질문이 없어서 (18.56%, 13,133건) ➔ 콘텐츠 리텐션을 유지할 질문 추천 기능 및 신규 질문 공급 필요.

- 기술적/비용 문제:

버그가 너무 많아서 (2.87%, 2,031건)

구독료가 너무 비싸서 (1.03%, 730건)

2️⃣ 위험 그룹별 제재/정지 유저 비율 (stg_users)
- High Risk (고신고/고차단): 제재 유저 비율이 1.93% (53명 / 2,742명)로 가장 높게 나타남.

- Heavy Hide User (숨김 다수): 1.28% (85명 / 6,644명)

- Normal (일반 유저): 1.28% (8,515명 / 667,699명)

💡 핵심 시사점: High Risk 그룹이 일반 유저보다 실제로 서비스 내 제재/정지 조치를 받을 확률이 약 1.5배 높다는 점이 입증되었습니다. 다만, 절대적인 수치(53명)는 전체 서비스 규모에 비해 크지 않으므로, 유저 제재보다는 마찰을 일으키는 질문 콘텐츠 관리에 초점을 맞추는 것이 더 효과적인 전략입니다.

## 3. High Risk 응답 비중이 높은 질문 키워드/패턴 분석

In [ ]:
import pandas as pd
from collections import Counter
from google.cloud import bigquery

client = bigquery.Client(project='your-gcp-project')

# 1. 질문별 High Risk 유저 응답 비중 추출
query_question_keywords = """
WITH user_groups AS (
    SELECT
        user_id,
        CASE
            WHEN (COALESCE(report_count, 0) + COALESCE(block_count, 0)) >= 3 THEN 'High Risk'
            WHEN COALESCE(hide_count, 0) >= 5 THEN 'Heavy Hide User'
            ELSE 'Normal'
        END AS user_group
    FROM `your-gcp-project.stage.stg_users`
)
SELECT
    q.question_id,
    q.question_text,
    COUNT(CASE WHEN ug.user_group = 'High Risk' THEN 1 END) AS high_risk_votes,
    COUNT(v.vote_id) AS total_votes
FROM `your-gcp-project.stage.stg_question` q
INNER JOIN `your-gcp-project.stage.stg_vote` v ON q.question_id = v.question_id
INNER JOIN user_groups ug ON v.sender_user_id = ug.user_id
GROUP BY 1, 2
HAVING total_votes >= 50
"""

df_q = client.query(query_question_keywords).to_dataframe()
df_q['high_risk_ratio'] = (df_q['high_risk_votes'] / df_q['total_votes']) * 100

# High Risk 비중 상위 20% 질문 vs 하위 질문 추출
top_risk_questions = df_q.sort_values(by='high_risk_ratio', ascending=False).head(30)

# 2. 질문 텍스트에서 주요 형태소/단어 추출 및 빈도 계산
words = []
for text in top_risk_questions['question_text'].dropna():
    # 2글자 이상의 단어만 추출
    tokens = [w for w in text.split() if len(w) >= 2]
    words.extend(tokens)

word_counts = Counter(words).most_common(15)
df_words = pd.DataFrame(word_counts, columns=['주요 키워드', '출현 빈도'])

print("=== 🚨 High Risk 응답 비중 상위 질문의 주요 등장 키워드 Top 15 ===")
display(df_words)

print("\n=== 🔝 High Risk 비중 상위 질문 Top 10 예시 ===")
display(top_risk_questions[['question_text', 'high_risk_votes', 'total_votes', 'high_risk_ratio']].head(10))

**High Risk 유저가 반응하는 마찰 유발 질문 키워드**
- **핵심 질문 특징:** 어떤 일로 멀어졌는데 다시 관계 회복하고 싶은 사람은? (4.08%), 전화번호 얻고 싶은 사람은? (2.80%), 키스하고 싶은 사람은? (2.65%), 돈 빌려줄 수 있는 친구 (2.60%) 등 개인적인 민감한 관계, 호감, 금전/마찰 유발 주제에서 High Risk 유저 비중이 유독 높게 나타남.

- **주요 단어:** 사람은?, 같은, 가장, 친구는? 등 타인을 특정 지목해야 하는 질문 패턴에서 마찰이 자주 발생함.

## 4. 유저 위험 그룹별 이용 행동 (투표 수, 결제, 활동성) 비교 분석

In [ ]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project='your-gcp-project')

query_user_behavior = """
WITH user_groups AS (
    SELECT
        user_id,
        CASE
            WHEN (COALESCE(report_count, 0) + COALESCE(block_count, 0)) >= 3 THEN 'High Risk (고신고/고차단)'
            WHEN COALESCE(hide_count, 0) >= 10 THEN 'Heavy Hide User (숨김 다수)'
            ELSE 'Normal (일반)'
        END AS risk_group,
        report_count,
        block_count,
        hide_count
    FROM `your-gcp-project.stage.stg_users`
    WHERE is_valid_user = TRUE
),
vote_stats AS (
    SELECT
        sender_user_id AS user_id,
        COUNT(vote_id) AS total_votes_sent
    FROM `your-gcp-project.stage.stg_vote`
    GROUP BY 1
)
SELECT
    ug.risk_group,
    COUNT(ug.user_id) AS user_count,
    -- 평균 투표 참여 건수
    ROUND(AVG(COALESCE(vs.total_votes_sent, 0)), 1) AS avg_votes_sent,
    -- 중앙값 투표 건수
    APPROX_QUANTILES(COALESCE(vs.total_votes_sent, 0), 100)[OFFSET(50)] AS median_votes_sent,
    -- 평균 신고받은 건수
    ROUND(AVG(COALESCE(ug.report_count, 0)), 2) AS avg_report_count,
    -- 평균 차단당한 건수
    ROUND(AVG(COALESCE(ug.block_count, 0)), 2) AS avg_block_count,
    -- 평균 숨긴 건수
    ROUND(AVG(COALESCE(ug.hide_count, 0)), 2) AS avg_hide_count
FROM user_groups ug
LEFT JOIN vote_stats vs ON ug.user_id = vs.user_id
GROUP BY 1
ORDER BY user_count DESC;
"""

df_behavior = client.query(query_user_behavior).to_dataframe()

print("=== 📊 유저 위험 그룹별 활동성(투표/신고/차단/숨김) 비교 ===")
display(df_behavior)

**유저 위험 그룹별 이용 활동성 비교**
- **투표 참여도:** High Risk 유저는 평균 3.6건의 투표를 보내 일반 유저(1.8건)나 Heavy Hide 유저(2.7건)보다 2배 가까이 높은 헤비 헤비 이용자 특성을 보임.

- **활동 양상:** High Risk 유저는 서비스 이용률(투표 참여) 자체가 매우 높기 때문에 타인과의 마찰(신고 3.27회, 차단 2.19회)에 노출될 확률도 함께 증가하는 것으로 확인됨.

### 인원수 불균형을 극복하는 '상세 활동성 비교'
이 코드는 단순 평균뿐만 아니라 1) 실제 1회 이상 투표한 '활동 유저' 비율, 2) 활동 유저만의 평균 투표 수, 3) 상위 10%/25%/50% 분위수 비교

In [ ]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project='your-gcp-project')

# 1. BigQuery에서는 기본 집계(전체 유저수, 활동 유저수, 평균 등)만 수행
query_robust_behavior = """
WITH user_groups AS (
    SELECT
        user_id,
        CASE
            WHEN COALESCE(ban_status, 'N') != 'N'
                 OR (COALESCE(report_count, 0) + COALESCE(block_count, 0)) >= 3
                 THEN 'High Risk (고신고/고차단)'
            WHEN COALESCE(hide_count, 0) >= 10
                 AND (COALESCE(report_count, 0) + COALESCE(block_count, 0)) < 3
                 THEN 'Heavy Hide User (숨김 다수 사용)'
            ELSE 'Normal (일반)'
        END AS risk_group
    FROM `your-gcp-project.stage.stg_users`
    WHERE is_valid_user = TRUE
),
vote_stats AS (
    SELECT
        sender_user_id AS user_id,
        COUNT(vote_id) AS total_votes_sent
    FROM `your-gcp-project.stage.stg_vote`
    GROUP BY 1
)
SELECT
    ug.risk_group,
    ug.user_id,
    COALESCE(vs.total_votes_sent, 0) AS total_votes_sent
FROM user_groups ug
LEFT JOIN vote_stats vs ON ug.user_id = vs.user_id
"""

# 전체 유저별 투표 데이터 로드
df_raw = client.query(query_robust_behavior).to_dataframe()

# 2. Pandas를 이용해 그룹별 집계 및 활성 유저 기준 분위수 계산
summary_list = []
for group, group_df in df_raw.groupby('risk_group'):
    total_users = len(group_df)
    active_df = group_df[group_df['total_votes_sent'] > 0]
    active_users = len(active_df)
    active_pct = round(active_users * 100.0 / total_users, 2) if total_users > 0 else 0

    avg_all = round(group_df['total_votes_sent'].mean(), 1)
    avg_active = round(active_df['total_votes_sent'].mean(), 1) if active_users > 0 else 0

    # 활성 유저 기준 분위수 (데이터가 없으면 0)
    median_val = round(active_df['total_votes_sent'].quantile(0.5), 1) if active_users > 0 else 0
    p75_val = round(active_df['total_votes_sent'].quantile(0.75), 1) if active_users > 0 else 0
    p90_val = round(active_df['total_votes_sent'].quantile(0.9), 1) if active_users > 0 else 0

    summary_list.append({
        '위험그룹': group,
        '전체유저수': total_users,
        '투표참여유저수': active_users,
        '실제활동유저비율(%)': active_pct,
        '전체기준_평균투표수': avg_all,
        '활동유저전용_평균투표수': avg_active,
        '활동유저_중앙값(Median)': median_val,
        '활동유저_상위25%(P75)': p75_val,
        '활동유저_상위10%(P90)': p90_val
    })

df_robust = pd.DataFrame(summary_list).sort_values(by='전체유저수', ascending=False)

print("=== 📊 인원수 불균형을 고려한 그룹별 실제 투표 활동성 비교 (Pandas 후처리 완료) ===")
display(df_robust)

## 5.성별(Gender) 분포 분석

In [ ]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project='your-gcp-project')

# 위험 그룹별 성별 분포 집계
query_gender = """
WITH user_groups AS (
    SELECT
        user_id,
        gender,
        CASE
            WHEN (COALESCE(report_count, 0) + COALESCE(block_count, 0)) >= 3 THEN 'High Risk (고신고/고차단)'
            WHEN COALESCE(hide_count, 0) >= 10 THEN 'Heavy Hide User (숨김 다수)'
            ELSE 'Normal (일반)'
        END AS risk_group
    FROM `your-gcp-project.stage.stg_users`
    WHERE is_valid_user = TRUE
)
SELECT
    risk_group,
    COALESCE(gender, '미선택/기타') AS gender,
    COUNT(user_id) AS user_count,
    ROUND(COUNT(user_id) * 100.0 / SUM(COUNT(user_id)) OVER(PARTITION BY risk_group), 2) AS ratio_pct
FROM user_groups
GROUP BY 1, 2
ORDER BY risk_group, user_count DESC;
"""

df_gender = client.query(query_gender).to_dataframe()

print("=== 🚻 위험 그룹별 성별(Gender) 분포 ===")
display(df_gender)

## 6. 월별/일별 신고·차단·숨김 발생 추이 (언제 많이 일어났나?) 뭔가 망한듯

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery

client = bigquery.Client(project='your-gcp-project')

# 유저별 최초 투표 월(활동 시작월) 기준 위험 그룹 추이 분석
query_user_timeline = """
WITH user_first_activity AS (
    SELECT
        sender_user_id AS user_id,
        DATE_TRUNC(EXTRACT(DATE FROM MIN(voted_at)), MONTH) AS first_active_month
    FROM `your-gcp-project.stage.stg_vote`
    GROUP BY 1
),
user_groups AS (
    SELECT
        u.user_id,
        CASE
            WHEN (COALESCE(u.report_count, 0) + COALESCE(u.block_count, 0)) >= 3 THEN 'High Risk'
            WHEN COALESCE(u.hide_count, 0) >= 10 THEN 'Heavy Hide User'
            ELSE 'Normal'
        END AS risk_group,
        fa.first_active_month
    FROM `your-gcp-project.stage.stg_users` u
    LEFT JOIN user_first_activity fa ON u.user_id = fa.user_id
    WHERE u.is_valid_user = TRUE
)
SELECT
    COALESCE(first_active_month, DATE '1970-01-01') AS active_month,
    risk_group,
    COUNT(user_id) AS user_count
FROM user_groups
WHERE first_active_month IS NOT NULL -- 실제 투표 이력이 있는 유저 대상
GROUP BY 1, 2
ORDER BY active_month, risk_group;
"""

df_timeline = client.query(query_user_timeline).to_dataframe()

# 시각화 (선 그래프)
plt.figure(figsize=(12, 5))
sns.lineplot(data=df_timeline, x='active_month', y='user_count', hue='risk_group', marker='o', linewidth=2.5)

plt.title('월별 위험 그룹 유저의 첫 활동(투표) 유입 추이', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('첫 활동 월 (Month)', fontsize=12)
plt.ylabel('유저 수', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("=== 📅 월별 위험 그룹 유저 첫 활동 건수 표 ===")
df_pivot_time = df_timeline.pivot_table(index='active_month', columns='risk_group', values='user_count', fill_value=0)
display(df_pivot_time)

1️⃣ "가입만 하고 투표 안 한 유저가 99% 이상" (극심한 롱테일 분포)
- 놀라운 사실: 전체 유저 중 실제로 투표를 1회 이상 참여해 본 유저 비율(실제활동유저비율)은 모든 그룹에서 1% 미만(0.67% ~ 0.93%)에 불과합니다.

- 의미: 전체 유저의 99% 이상이 투표를 한 한 번도 하지 않은 '유령/비활동 회원'이기 때문에, 중앙값(Median), 상위 25%, 상위 10% 수치가 모두 0으로 잡힌 것입니다.

2️⃣ "전체 기준 평균" vs "실제 활동 유저 전용 평균"의 반전
전체 유저 기준 평균:

- Normal: 1.8건 vs High Risk: 3.6건

    - (해석: High Risk 유저가 2배 더 많이 투표하는 것처럼 보임)

- 실제 활동 유저(1회 이상 투표자) 전용 평균:

    - Normal: 249.2건

    - Heavy Hide User: 395.6건

    - High Risk: 390.8건


- 3️⃣ 최종 결론 (보고서 및 피치덱용 멘트)
"High Risk 유저가 단순히 투표를 더 많이 해서 마찰이 일어난 것이 아닙니다."

- 전체 유저 평균만 보았을 때는 High Risk 그룹의 투표 수가 2배 높게 착시가 일어났으나, 이는 Normal 그룹에 비활동 유저가 99% 이상 포함되어 있었기 때문입니다.

- 실제 투표 활동을 활발히 하는 헤비 유저들끼리 비교했을 때, Heavy Hide(395.6건)나 High Risk(390.8건) 유저뿐만 아니라 Normal 그룹의 헤비 유저 역시 평균 249.2건이라는 엄청난 양의 투표를 보내고 있습니다.

- 따라서 위험 그룹의 마찰 원인은 '단순히 투표를 많이 해서'가 아니라, '어떤 특정 질문(민감한 관계/금전/호감 질문)에 집중적으로 참여하거나 행동하는 양상' 때문인 것으로 해석할 수 있습니다.

## 7. 학교급별 위험 그룹 분포 확인용

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery

client = bigquery.Client(project='your-gcp-project')

# 유저 -> 학급(accounts_group) -> 학교(accounts_school) 조인을 통한 학교급별 위험 그룹 분포 쿼리
query_school_grade = """
WITH user_groups AS (
    SELECT
        u.user_id,
        CASE
            WHEN s.school_type = 'M' THEN '중학교 (Middle School)'
            WHEN s.school_type = 'H' THEN '고등학교 (High School)'
            ELSE '기타/미분류'
        END AS school_grade,
        CASE
            WHEN (COALESCE(u.report_count, 0) + COALESCE(u.block_count, 0)) >= 3 THEN 'High Risk (고신고/고차단)'
            WHEN COALESCE(u.hide_count, 0) >= 10 THEN 'Heavy Hide User (숨김 다수)'
            ELSE 'Normal (일반)'
        END AS risk_group
    FROM `your-gcp-project.stage.stg_users` u
    LEFT JOIN `your-gcp-project.stage.stg_group` g ON u.group_id = g.group_id
    LEFT JOIN `your-gcp-project.stage.stg_school` s ON g.school_id = s.school_id
    WHERE u.is_valid_user = TRUE
)
SELECT
    school_grade,
    risk_group,
    COUNT(user_id) AS user_count
FROM user_groups
GROUP BY 1, 2
ORDER BY school_grade, risk_group;
"""

df_school = client.query(query_school_grade).to_dataframe()

print("=== 🏫 중·고등학교별 위험 그룹 분포 표 ===")
df_pivot_school = df_school.pivot_table(index='school_grade', columns='risk_group', values='user_count', fill_value=0)
display(df_pivot_school)

📊 중·고등학교별 위험 그룹 분포 요약
- 고등학교 (High School)

    - Normal (일반): 366,163명

    - Heavy Hide User (숨김 다수): 3,471명

    - High Risk (고신고/고차단): 1,402명

- 중학교 (Middle School)

    - Normal (일반): 292,932명

    - Heavy Hide User (숨김 다수): 3,088명

    - High Risk (고신고/고차단): 1,282명

💡 인사이트 제안
전체 유저 규모 대비 각 위험 그룹이 차지하는 비중(비율)을 살펴보면 중·고등학생 간 성향 차이를 더 깊이 있게 분석하실 수 있습니다.

인원 수 불균형을 보정하기 위해 다음 단계로 비율(%)을 계산하는 시각화나 상세 비교 분석

In [ ]:
%%bigquery --project your-gcp-project
-- 1. 전체 유저 수 및 중복/결측치 통합 진단 쿼리
WITH base_data AS (
  SELECT *
  FROM `your-gcp-project.stage.stg_users` -- 필요시 실제 데이터셋/테이블 경로로 수정해주세요
)
SELECT
    '1. 전체 행 수' AS check_item,
    CAST(COUNT(*) AS STRING) AS result_value
FROM base_data

UNION ALL

SELECT
    '2. 고유 유저 수 (user_id 기준)' AS check_item,
    CAST(COUNT(DISTINCT user_id) AS STRING) AS result_value
FROM base_data

UNION ALL

SELECT
    '3. 중복된 유저 ID 수 (중복 행 발생 여부)' AS check_item,
    CAST(COUNT(*) - COUNT(DISTINCT user_id) AS STRING) AS result_value
FROM base_data

UNION ALL

SELECT
    '4. user_id 결측치 수' AS check_item,
    CAST(COUNTIF(user_id IS NULL) AS STRING) AS result_value
FROM base_data;